In [1]:
!git clone https://github.com/sneefyyy/DSL.git

Cloning into 'DSL'...
remote: Enumerating objects: 203, done.
remote: Counting objects: 100% (203/203), done.
remote: Compressing objects: 100% (116/116), done.
remote: Total 203 (delta 91), reused 176 (delta 64), pack-reused 0 (from 0)
Receiving objects: 100% (203/203), 3.08 MiB | 14.62 MiB/s, done.
Resolving deltas: 100% (91/91), done.


In [2]:
!pip install transformers peft datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 70.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 32.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 73.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 67.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.7/42.7 MB 48.2 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 789.9/789.9 kB 44.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 59.4 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25/25 [datasets]/25 [datasets]e]s]ub]


In [3]:
!pip install --upgrade transformers

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_repo = "Qwen/Qwen2.5-Coder-7B-Instruct"
adapter_repo = "middles/qwen-7b-arc-with-eval-v2"

tokenizer = AutoTokenizer.from_pretrained(base_repo)
model = AutoModelForCausalLM.from_pretrained(base_repo, device_map="cuda")
model = PeftModel.from_pretrained(model, adapter_repo)

def generate(model, tokenizer, prompt, max_new_tokens=64, skip_special_tokens=False):
    tokenized_input = tokenizer(
        prompt, add_special_tokens=False, return_tensors="pt"
    ).to(model.device)

    model.eval()
    gen_output = model.generate(**tokenized_input,
                                eos_token_id=tokenizer.eos_token_id,
                                max_new_tokens=max_new_tokens)

    output = tokenizer.batch_decode(gen_output, skip_special_tokens=skip_special_tokens)
    return output[0]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

...

model-00002-of-00004.safetensors:   0%|          | 0.00/4.93G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/53.2M [00:00<?, ?B/s]

In [5]:
prompt = """<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write Python code that transforms test_input into the expected test output. Only output executable Python lines setting output_grid.
Training Example 1:
Input: [[1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
Output: [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 1]]

Training Example 2:
Input: [[6, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
Output: [[6, 0, 0, 0, 0], [0, 6, 0, 0, 0], [0, 0, 6, 0, 0], [0, 0, 0, 6, 0], [0, 0, 0, 0, 6]]

Test Input: [[7, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
Test Output: <|im_end|>
<|im_start|>assistant"""
out = model.generate(**tokenizer(prompt, return_tensors="pt").to(model.device), max_new_tokens=128)
print(tokenizer.decode(out[0], skip_special_tokens=False))
output = tokenizer.decode(out[0], skip_special_tokens=False)

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write Python code that transforms test_input into the expected test output. Only output executable Python lines setting output_grid.
Training Example 1:
Input: [[1, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
Output: [[1, 0, 0, 0, 0], [0, 1, 0, 0, 0], [0, 0, 1, 0, 0], [0, 0, 0, 1, 0], [0, 0, 0, 0, 1]]

Training Example 2:
Input: [[6, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
Output: [[6, 0, 0, 0, 0], [0, 6, 0, 0, 0], [0, 0, 6, 0, 0], [0, 0, 0, 6, 0], [0, 0, 0, 0, 6]]

Test Input: [[7, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
Test Output: <|im_end|>
<|im_start|>assistant
output_grid = RepeatPatternGenerator.repeat_diagonal(test_input, 5)<|im_end|>


## Testing with Interpreter

In [6]:

import os
import re
import json
import time
import argparse
import traceback
import inspect
from typing import Dict, Any, Optional

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig

# ---------------------------------------------------------------------
# Extract code after <|assistant|> up to <|end|>
# ---------------------------------------------------------------------
ASSISTANT_BLOCK_RE = re.compile(r"<\|assistant\|>(.*?)(?:<\|end\|>|$)", re.DOTALL)

def extract_output_grid_line(full_text: str) -> str:
    match = ASSISTANT_BLOCK_RE.findall(full_text)
    if not match:
        return ""
    assistant_segment = match[-1]
    for line in assistant_segment.strip().splitlines():
        if "output_grid" in line:
            return line.strip()
    return assistant_segment.strip()

In [7]:
import re

# Updated regex patterns for different formats
ASSISTANT_BLOCK_RE = re.compile(r"<\|assistant\|>(.*?)(?:<\|end\|>|$)", re.DOTALL)
IM_ASSISTANT_BLOCK_RE = re.compile(r"<\|im_start\|>assistant(.*?)(?:<\|im_end\|>|$)", re.DOTALL)

def extract_output_grid_line(full_text: str) -> str:
    """
    Extract the output_grid line from model output.
    Handles both <|assistant|> and <|im_start|>assistant formats.
    """
    # Try the im_start format first (Qwen style)
    match = IM_ASSISTANT_BLOCK_RE.findall(full_text)

    # If not found, try the original format
    if not match:
        match = ASSISTANT_BLOCK_RE.findall(full_text)

    if not match:
        return ""

    # Get the last assistant segment
    assistant_segment = match[-1]

    # Look for output_grid assignment
    for line in assistant_segment.strip().splitlines():
        if "output_grid" in line:
            return line.strip()

    # If no specific output_grid line, return the whole segment
    return assistant_segment.strip()

# Test with your example
test_text = """<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write Python code that transforms test_input into the expected test output. Only output executable Python lines setting output_grid.
Training Example 1:
Input: [[0, 0, 0, 0, 0], [0, 8, 0, 8, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
Output: [[0, 0, 0, 0, 0], [0, 8, 8, 8, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]

Training Example 2:
Input: [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 9], [0, 0, 0, 0, 0], [0, 0, 9, 0, 0]]
Output: [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 9], [0, 0, 0, 9, 0], [0, 0, 9, 0, 0]]

Test Input: [[0, 0, 0, 7, 0], [0, 0, 0, 0, 7], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
Test Output: <|im_end|>
<|im_start|>assistant
output_grid = ConnectGenerator.connect(test_input, (1, 4), (0, 3), 7)<|im_end|>"""

result = extract_output_grid_line(test_text)
print(f"Extracted: {result}")
# Output: Extracted: output_grid = ConnectGenerator.connect(test_input, (1, 4), (0, 3), 7)

Extracted: output_grid = ConnectGenerator.connect(test_input, (1, 4), (0, 3), 7)


## Retrieve testing data

In [8]:
test_dataset = load_dataset("middles/dsl-chained-pipeline-v0.1.0", split="train")
test_dataset

README.md: 0.00B [00:00, ?B/s]

Repo card metadata block was not found. Setting CardData to empty.


data/train-00000-of-00001.parquet:   0%|          | 0.00/27.3k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/11.0k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/11.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/240 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/30 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/30 [00:00<?, ? examples/s]

Dataset({
    features: ['train_input1', 'train_output1', 'train_input2', 'train_output2', 'test_input', 'test_output', 'solution', 'transform_chain', 'chain_length'],
    num_rows: 240
})

In [9]:
import json
from datasets import load_dataset


def format_row(ex):
    prompt = (
        "Write Python code that transforms test_input into the expected test output. Only output executable Python lines setting output_grid. You may chain together multiple functions, but the last assignment MUST be output_grid.\n"
        "Training Example 1:\n"
        f"Input: {json.dumps(ex['train_input1'])}\n"
        f"Output: {json.dumps(ex['train_output1'])}\n\n"
        "Training Example 2:\n"
        f"Input: {json.dumps(ex['train_input2'])}\n"
        f"Output: {json.dumps(ex['train_output2'])}\n\n"
        f"Test Input: {json.dumps(ex['test_input'])}\n"
        "Test Output: "
    )
    sol = ex.get("solution", "")
    if isinstance(sol, (list, tuple)):
        completion = "\n".join(str(s) for s in sol)
    else:
        completion = str(sol).strip()
    return {
        "prompt": prompt,
        "completion": completion,
        "full_text": prompt + completion,
    }

# Apply
test_dataset = test_dataset.map(format_row, desc="Formatting prompt/completion")
# Drop raw columns if no longer needed
DROP_COLS = [
    "train_input1","train_output1","train_input2","train_output2",
    "test_input","test_output","solution"
]
test_dataset = test_dataset.remove_columns(DROP_COLS)

# Single example demonstration
example = {
    'train_input1': [[0,0,0,0,0],[0,0,0,7,0],[0,0,0,0,7],[0,0,0,0,0],[0,0,0,0,0]],
    'train_output1': [[0,0,0,0,0],[0,0,0,7,0],[0,0,0,0,7],[0,0,0,0,0],[0,0,0,0,0]],
    'train_input2': [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,3],[0,0,0,0,3]],
    'train_output2': [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,3],[0,0,0,0,3]],
    'test_input': [[0,0,0,0,0],[0,0,0,0,0],[0,0,0,0,0],[0,5,0,0,0],[0,0,5,0,0]],
    'solution': "output_grid = ConnectGenerator.connect(test_input, (3,1), (4,2), 5)"
}
formatted = format_row(example)
print(formatted['prompt'])
print(formatted['completion'])

Formatting prompt/completion:   0%|          | 0/240 [00:00<?, ? examples/s]

Write Python code that transforms test_input into the expected test output. Only output executable Python lines setting output_grid. You may chain together multiple functions, but the last assignment MUST be output_grid.
Training Example 1:
Input: [[0, 0, 0, 0, 0], [0, 0, 0, 7, 0], [0, 0, 0, 0, 7], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
Output: [[0, 0, 0, 0, 0], [0, 0, 0, 7, 0], [0, 0, 0, 0, 7], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]

Training Example 2:
Input: [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 3], [0, 0, 0, 0, 3]]
Output: [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 3], [0, 0, 0, 0, 3]]

Test Input: [[0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 5, 0, 0, 0], [0, 0, 5, 0, 0]]
Test Output: 
output_grid = ConnectGenerator.connect(test_input, (3,1), (4,2), 5)


In [10]:
def format_dataset(examples):
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role": "user", "content": examples["prompt"][i]},
                {"role": "assistant", "content": examples["completion"][i]},
            ]
            output_texts.append(converted_sample)
        return {'messages': output_texts}
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return {'messages': converted_sample}

test_dataset = test_dataset.map(format_dataset).remove_columns(['prompt', 'completion'])
test_dataset[0]['messages']

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

[{'content': 'Write Python code that transforms test_input into the expected test output. Only output executable Python lines setting output_grid. You may chain together multiple functions, but the last assignment MUST be output_grid.\nTraining Example 1:\nInput: [[1, 9, 1, 6, 0], [0, 7, 0, 0, 1], [8, 1, 3, 1, 7], [0, 1, 0, 8, 9], [0, 0, 9, 0, 0]]\nOutput: [[0, 0, 1, 0, 1], [0, 1, 1, 1, 1], [1, 0, 1, 0, 1], [1, 1, 1, 0, 1], [1, 1, 1, 1, 0]]\n\nTraining Example 2:\nInput: [[0, 3, 0, 8, 0], [6, 0, 0, 0, 0], [0, 2, 0, 6, 0], [0, 0, 0, 5, 0], [0, 0, 8, 0, 9]]\nOutput: [[0, 0, 0, 3, 0], [0, 0, 3, 0, 3], [3, 0, 0, 0, 0], [0, 3, 3, 0, 3], [3, 0, 0, 0, 0]]\n\nTest Input: [[4, 6, 0, 0, 0], [2, 0, 0, 1, 0], [0, 0, 0, 6, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]\nTest Output: ',
  'role': 'user'},
 {'content': 'step_1 = FloodFillGenerator.flood_fill(test_input, (0, 4), 7)\noutput_grid = RotateShapeGenerator.rotate_clockwise(step_1)',
  'role': 'assistant'}]

In [11]:
tokenizer.chat_template

'{%- if tools %}\n    {{- \'<|im_start|>system\\n\' }}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- messages[0][\'content\'] }}\n    {%- else %}\n        {{- \'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.\' }}\n    {%- endif %}\n    {{- "\\n\\n# Tools\\n\\nYou may call one or more functions to assist with the user query.\\n\\nYou are provided with function signatures within <tools></tools> XML tags:\\n<tools>" }}\n    {%- for tool in tools %}\n        {{- "\\n" }}\n        {{- tool | tojson }}\n    {%- endfor %}\n    {{- "\\n</tools>\\n\\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\\n<tool_call>\\n{\\"name\\": <function-name>, \\"arguments\\": <args-json-object>}\\n</tool_call><|im_end|>\\n" }}\n{%- else %}\n    {%- if messages[0][\'role\'] == \'system\' %}\n        {{- \'<|im_start|>system\\n\' + messages[0][\'content\'] + \'<|im_end|>\\n\' }}\n    {%- else %}\n       

In [12]:
print(tokenizer.apply_chat_template(test_dataset[5]['messages'], tokenize=False))

<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write Python code that transforms test_input into the expected test output. Only output executable Python lines setting output_grid. You may chain together multiple functions, but the last assignment MUST be output_grid.
Training Example 1:
Input: [[3, 0, 8, 6, 1], [3, 0, 6, 9, 0], [0, 0, 3, 0, 4], [0, 4, 0, 0, 0], [0, 0, 0, 8, 0]]
Output: [[3, 0, 8, 6, 7], [3, 0, 6, 9, 0], [0, 0, 3, 0, 4], [0, 4, 7, 7, 0], [7, 0, 0, 8, 0]]

Training Example 2:
Input: [[0, 8, 5, 3, 0], [0, 0, 0, 0, 0], [0, 2, 0, 0, 0], [0, 0, 0, 4, 4], [0, 1, 0, 0, 2]]
Output: [[0, 8, 5, 3, 0], [0, 0, 0, 0, 0], [6, 2, 0, 0, 0], [0, 0, 0, 6, 4], [0, 6, 0, 0, 2]]

Test Input: [[7, 0, 0, 5, 0], [9, 9, 0, 3, 0], [0, 0, 0, 0, 2], [0, 7, 0, 0, 2], [0, 0, 0, 0, 0]]
Test Output: <|im_end|>
<|im_start|>assistant
step_1 = ExtractPatternGenerator.extract_non_background(test_input)
output_grid = CreateShapeGenerator.c

In [13]:
test_dataset_practice = tokenizer.apply_chat_template(test_dataset[5]['messages'], tokenize=False)

In [14]:
extract_output_grid_line(test_dataset_practice)

'output_grid = CreateShapeGenerator.create_shape([(3, 2), (2, 1), (2, 0)], 1, step_1)'

## Interpreter

In [15]:
!cd DSL

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [16]:
import json
import re
import copy
import traceback
import inspect
from typing import Dict, List, Any, Optional, Tuple

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# Import your generators - assuming they're in a 'generators' module
from DSL.generators import *

In [17]:
class ModelOutputInterpreter:
    """
    Interprets and tests model-generated solutions for ARC tasks.
    """

    def __init__(self):
        """Initialize the interpreter with generator classes."""
        self.generator_classes = self.discover_generator_classes()
        self.namespace_template = self.create_execution_namespace()
        # Regex to extract code after <|assistant|> tag
        self.assistant_pattern = re.compile(r'<\|assistant\|>(.*?)(?:<\|end\|>|$)', re.DOTALL)

    def discover_generator_classes(self):
        """
        Discover all available generator classes from the generators module.
        """
        generator_classes = {}

        # Get all items from generators module
        import DSL.generators
        for name in dir(DSL.generators):
            obj = getattr(DSL.generators, name)
            if inspect.isclass(obj) and name.endswith('Generator'):
                generator_classes[name] = obj

        return generator_classes

    def create_execution_namespace(self):
        """
        Create a namespace with all necessary imports and classes for execution.
        This matches your original interpreter's namespace creation.
        """
        namespace = {
            'copy': copy,
            '__builtins__': __builtins__,
        }

        # Add all discovered generator classes
        namespace.update(self.generator_classes)

        # Create instances for non-base generators
        for class_name, class_obj in self.generator_classes.items():
            if class_name != 'ExampleGenerator':
                try:
                    instance = class_obj()
                    # Store with a lowercase name for convenience
                    instance_name = class_name[0].lower() + class_name[1:]
                    namespace[instance_name] = instance
                except:
                    pass

        # Add direct function aliases for common method names
        for class_name, class_obj in self.generator_classes.items():
            try:
                for attr in dir(class_obj):
                    if attr.startswith('_'):
                        continue
                    try:
                        attr_obj = getattr(class_obj, attr)
                        if callable(attr_obj):
                            # Add the raw method name as a top-level name
                            if attr not in namespace:
                                namespace[attr] = attr_obj

                            # Add a variant where the first token is capitalized
                            parts = attr.split('_')
                            if len(parts) > 1:
                                cap_variant = '_'.join([parts[0].capitalize()] + parts[1:])
                                if cap_variant not in namespace:
                                    namespace[cap_variant] = attr_obj
                    except Exception:
                        continue
            except Exception:
                continue

        return namespace

    def extract_solution_code(self, model_output: str) -> str:
        """
        Extract the solution code from model output.
        """
        match = self.assistant_pattern.findall(model_output)
        if not match:
            return ""

        assistant_segment = match[-1].strip()

        # Look for output_grid assignment
        for line in assistant_segment.splitlines():
            if "output_grid" in line:
                return line.strip()

        # If no specific output_grid line, return the whole segment
        return assistant_segment

    def test_solution(self,
                     solution_code: str,
                     test_input: List[List[int]],
                     expected_output: List[List[int]]) -> Tuple[bool, Optional[List[List[int]]], str]:
        """
        Test a solution code against expected output.
        """
        # Create a fresh namespace for this execution
        namespace = self.namespace_template.copy()
        namespace['test_input'] = copy.deepcopy(test_input)

        try:
            # Execute the solution
            exec(solution_code, namespace)

            # Get the output
            output_grid = namespace.get('output_grid')

            if output_grid is None:
                return False, None, "No output_grid variable found in solution"

            # Compare with expected
            success = output_grid == expected_output
            return success, output_grid, "" if success else "Output doesn't match expected"

        except Exception as e:
            error_msg = f"Error executing solution: {str(e)}"
            return False, None, error_msg

    def print_grid_comparison(self, grid1, grid2, title1="Grid 1", title2="Grid 2"):
        """Print two grids side by side for comparison."""
        print(f"\n{title1:<20} {title2:<20}")
        print("-" * 41)

        max_rows = max(len(grid1), len(grid2)) if grid1 and grid2 else 0
        for i in range(max_rows):
            # Row from grid1
            if grid1 and i < len(grid1):
                row1 = " ".join(str(cell) if cell != 0 else "." for cell in grid1[i])
            else:
                row1 = ""

            # Row from grid2
            if grid2 and i < len(grid2):
                row2 = " ".join(str(cell) if cell != 0 else "." for cell in grid2[i])
            else:
                row2 = ""

            print(f"{row1:<20} {row2:<20}")

# Create interpreter instance
interpreter = ModelOutputInterpreter()
print(f"Discovered {len(interpreter.generator_classes)} generator classes:")
for name in interpreter.generator_classes:
    print(f"  - {name}")

Discovered 13 generator classes:
  - ChainedTransformGenerator
  - ConnectGenerator
  - CountAndTransformGenerator
  - CreateShapeGenerator
  - ExampleGenerator
  - ExtractPatternGenerator
  - FloodFillGenerator
  - MirrorShapeGenerator
  - MixedPairGenerator
  - MoveShapeGenerator
  - RepeatPatternGenerator
  - RotateShapeGenerator
  - SymmetryCompleteGenerator


In [57]:
# Load the test dataset
test_dataset = load_dataset("middles/dsl-chained-pipeline-v0.1.0", split="train")
print(f"Loaded {len(test_dataset)} test examples")
print(f"Dataset features: {test_dataset.features.keys()}")

# Show a sample
sample = test_dataset[0]
print(f"\nSample solution: {sample['solution']}")

Repo card metadata block was not found. Setting CardData to empty.


Loaded 240 test examples
Dataset features: dict_keys(['train_input1', 'train_output1', 'train_input2', 'train_output2', 'test_input', 'test_output', 'solution', 'transform_chain', 'chain_length'])

Sample solution: ['step_1 = FloodFillGenerator.flood_fill(test_input, (0, 4), 7)', 'output_grid = RotateShapeGenerator.rotate_clockwise(step_1)']


In [58]:
# Test on a single example
example_idx = 0  # Change this to test different examples
example = test_dataset[example_idx]

# Show the example
print(f"Testing example {example_idx}")
print(f"Expected solution: {example['solution']}")
print(f"Test input shape: {len(example['test_input'])}x{len(example['test_input'][0])}")
print(f"Test output shape: {len(example['test_output'])}x{len(example['test_output'][0])}")

Testing example 0
Expected solution: ['step_1 = FloodFillGenerator.flood_fill(test_input, (0, 4), 7)', 'output_grid = RotateShapeGenerator.rotate_clockwise(step_1)']
Test input shape: 5x5
Test output shape: 5x5


In [59]:
# Format dataset for prompt/completion structure
def format_row(ex):
    prompt = (
        "Write Python code that transforms test_input into the expected test output. Only output executable Python lines setting output_grid as the return variable. You may use multiple functions and intermediate steps.\n"
        "Training Example 1:\n"
        f"Input: {json.dumps(ex['train_input1'])}\n"
        f"Output: {json.dumps(ex['train_output1'])}\n\n"
        "Training Example 2:\n"
        f"Input: {json.dumps(ex['train_input2'])}\n"
        f"Output: {json.dumps(ex['train_output2'])}\n\n"
        f"Test Input: {json.dumps(ex['test_input'])}\n"
        "Test Output: "
    )
    sol = ex.get("solution", "")
    if isinstance(sol, (list, tuple)):
        completion = "\n".join(str(s) for s in sol)
    else:
        completion = str(sol).strip()
    return {
        "prompt": prompt,
        "completion": completion,
        "full_text": prompt + completion,
        # Keep original columns for reference
        "train_input1": ex['train_input1'],
        "train_output1": ex['train_output1'],
        "train_input2": ex['train_input2'],
        "train_output2": ex['train_output2'],
        "test_input": ex['test_input'],
        "test_output": ex['test_output'],
        "solution": ex['solution']
    }

# Apply formatting
test_dataset = test_dataset.map(format_row, desc="Formatting prompt/completion")
print(f"Formatted dataset with prompt/completion structure")

Formatting prompt/completion:   0%|          | 0/240 [00:00<?, ? examples/s]

Formatted dataset with prompt/completion structure


In [60]:
# Convert to chat format with roles
def format_dataset(examples):
    if isinstance(examples["prompt"], list):
        output_texts = []
        for i in range(len(examples["prompt"])):
            converted_sample = [
                {"role": "user", "content": examples["prompt"][i]},
                {"role": "assistant", "content": examples["completion"][i]},
            ]
            output_texts.append(converted_sample)
        return {'messages': output_texts}
    else:
        converted_sample = [
            {"role": "user", "content": examples["prompt"]},
            {"role": "assistant", "content": examples["completion"]},
        ]
        return {'messages': converted_sample}

# Apply chat format
test_dataset_formatted = test_dataset.map(format_dataset)
print(f"Converted to chat format")
print(f"\nExample message structure:")
print(f"User role: {test_dataset_formatted[0]['messages'][0]['role']}")
print(f"Assistant role: {test_dataset_formatted[0]['messages'][1]['role']}")

Map:   0%|          | 0/240 [00:00<?, ? examples/s]

Converted to chat format

Example message structure:
User role: user
Assistant role: assistant


In [61]:
# Test chat template application
example_chat = tokenizer.apply_chat_template(
    test_dataset_formatted[0]['messages'],
    tokenize=False
)
print("Chat template applied successfully!")
print(f"\nTemplate preview (first 500 chars):")
print(example_chat[:500] + "...")
print(f"\nTemplate preview (last 200 chars):")
print("..." + example_chat[-200:])

Chat template applied successfully!

Template preview (first 500 chars):
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Write Python code that transforms test_input into the expected test output. Only output executable Python lines setting output_grid as the return variable. You may use multiple functions and intermediate steps.
Training Example 1:
Input: [[1, 9, 1, 6, 0], [0, 7, 0, 0, 1], [8, 1, 3, 1, 7], [0, 1, 0, 8, 9], [0, 0, 9, 0, 0]]
Output: [[0, 0, 1, 0, 1], [0, 1, 1, 1, 1], [1, 0, 1, 0, 1], [...

Template preview (last 200 chars):
... 0], [0, 0, 0, 0, 0]]
Test Output: <|im_end|>
<|im_start|>assistant
step_1 = FloodFillGenerator.flood_fill(test_input, (0, 4), 7)
output_grid = RotateShapeGenerator.rotate_clockwise(step_1)<|im_end|>



In [62]:
# Generate solution with model
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
print(f"Generating solution...")

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=128,
        temperature=0.1,
        do_sample=True,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

model_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
print("\nModel output (last 300 chars):")
print("..." + model_output[-300:])

Generating solution...

Model output (last 300 chars):
...0, 0, 0], [0, 6, 0, 0, 0], [0, 0, 6, 0, 0], [0, 0, 0, 6, 0], [0, 0, 0, 0, 6]]

Test Input: [[7, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0], [0, 0, 0, 0, 0]]
Test Output: <|im_end|>
<|im_start|>assistant
output_grid = RepeatPatternGenerator.repeat_diagonal(test_input, 5)<|im_end|>


In [63]:
# Extract and test the solution
solution_code = extract_output_grid_line(model_output)
print(f"Extracted solution: {solution_code}")

# Test the solution
success, computed_output, error_msg = interpreter.test_solution(
    solution_code,
    example['test_input'],
    example['test_output']
)

print(f"\nResult: {'✓ PASSED' if success else '✗ FAILED'}")
if not success:
    print(f"Error: {error_msg}")

if computed_output is not None:
    interpreter.print_grid_comparison(
        example['test_output'],
        computed_output,
        "Expected", "Computed"
    )

Extracted solution: output_grid = RepeatPatternGenerator.repeat_diagonal(test_input, 5)

Result: ✗ FAILED
Error: Output doesn't match expected

Expected             Computed            
-----------------------------------------
4 4 4 4 4            4 . . . .           
4 4 4 4 4            . 4 . . .           
4 4 4 4 4            . . 4 . .           
4 4 4 4 4            . . . 4 .           
4 4 4 4 4            . . . . 4           


In [64]:
def evaluate_batch(model, tokenizer, dataset, dataset_formatted, start_idx=0, num_examples=10, verbose=True):
    """
    Evaluate model on a batch of examples.
    """
    results = []
    passed = 0

    end_idx = min(start_idx + num_examples, len(dataset))

    for idx in range(start_idx, end_idx):
        example = dataset[idx]
        example_messages = dataset_formatted[idx]['messages']

        if verbose:
            print(f"\n{'='*60}")
            print(f"Testing example {idx} ({idx-start_idx+1}/{num_examples})")
            print(f"Expected solution: {example['solution'][:100]}...")

        # Generate solution using chat template
        messages_for_generation = [example_messages[0]]  # Just user message
        prompt = tokenizer.apply_chat_template(
            messages_for_generation,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.1,
                do_sample=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id
            )

        model_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
        solution_code = extract_output_grid_line(model_output)

        if verbose:
            print(f"Generated: {solution_code}")

        # Test solution
        success, computed_output, error_msg = interpreter.test_solution(
            solution_code,
            example['test_input'],
            example['test_output']
        )

        if success:
            passed += 1
            if verbose:
                print("✓ PASSED")
        else:
            if verbose:
                print(f"✗ FAILED: {error_msg}")

        results.append({
            'idx': idx,
            'success': success,
            'generated_code': solution_code,
            'expected_solution': example['solution'],
            'error': error_msg if not success else None
        })

    accuracy = passed / num_examples if num_examples > 0 else 0

    print(f"\n{'='*60}")
    print(f"RESULTS: {passed}/{num_examples} tests passed")
    print(f"Accuracy: {accuracy:.2%}")

    return results, accuracy

In [65]:
import signal
from contextlib import contextmanager

@contextmanager
def timeout(duration):
    def timeout_handler(signum, frame):
        raise TimeoutError(f"Operation timed out after {duration} seconds")
    
    signal.signal(signal.SIGALRM, timeout_handler)
    signal.alarm(duration)
    try:
        yield
    finally:
        signal.alarm(0)

In [66]:
# Uncomment to run on full dataset (will take a while)
# from tqdm.auto import tqdm # Already imported tqdm in the previous cell
from tqdm.auto import tqdm

def evaluate_batch(model, tokenizer, dataset, dataset_formatted, start_idx=0, num_examples=10, verbose=True):
    """
    Evaluate model on a batch of examples.
    """

    
    
    results = []
    passed = 0

    end_idx = min(start_idx + num_examples, len(dataset))

    for idx in tqdm(range(start_idx, end_idx), desc="Evaluating batch"):
        if idx % 10 == 0:
            torch.cuda.empty_cache()
            
        example = dataset[idx]
        example_messages = dataset_formatted[idx]['messages']

        if verbose:
            print(f"\n{'='*60}")
            print(f"Testing example {idx} ({idx-start_idx+1}/{num_examples})")
            print(f"Expected solution: {example['solution'][:100]}...")

        # Generate solution using chat template
        messages_for_generation = [example_messages[0]]  # Just user message
        prompt = tokenizer.apply_chat_template(
            messages_for_generation,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

        try:
            with timeout(30):  # 30 second timeout per example
                with torch.no_grad():
                    outputs = model.generate(
                    **inputs,
                    max_new_tokens=128,
                    temperature=0.1,
                    do_sample=True,
                    eos_token_id=tokenizer.eos_token_id,
                    pad_token_id=tokenizer.pad_token_id
                )
        except TimeoutError:
            print(f"Example {idx} timed out, skipping...")
            continue

        model_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
        solution_code = extract_output_grid_line(model_output)
        
        del inputs, outputs
        torch.cuda.empty_cache()

        if verbose:
            print(f"Generated: {solution_code}")

        # Test solution
        success, computed_output, error_msg = interpreter.test_solution(
            solution_code,
            example['test_input'],
            example['test_output']
        )

        if success:
            passed += 1
            if verbose:
                print("✓ PASSED")
        else:
            if verbose:
                print(f"✗ FAILED: {error_msg}")

        results.append({
            'idx': idx,
            'success': success,
            'generated_code': solution_code,
            'expected_solution': example['solution'],
            'error': error_msg if not success else None
        })

    accuracy = passed / num_examples if num_examples > 0 else 0

    print(f"\n{'='*60}")
    print(f"RESULTS: {passed}/{num_examples} tests passed")
    print(f"Accuracy: {accuracy:.2%}")

    return results, accuracy

In [67]:
import threading
from contextlib import contextmanager
import torch
from tqdm.auto import tqdm

class TimeoutException(Exception):
    pass

@contextmanager
def timeout_context(seconds: int):
    fired = {'hit': False}
    def mark(): fired['hit'] = True
    timer = threading.Timer(seconds, mark)
    timer.start()
    try:
        yield
        if fired['hit']:
            raise TimeoutException(f"Operation timed out after {seconds}s")
    finally:
        timer.cancel()

def pretty_grid(grid):
    if grid is None:
        return "None"
    return "\n".join("".join(str(c) for c in row) for row in grid)

def diff_grids(expected, computed):
    if expected is None or computed is None:
        return "No diff (one is None)"
    rows = min(len(expected), len(computed))
    cols = min(len(expected[0]), len(computed[0]))
    mismatch_coords = []
    out = []
    for r in range(rows):
        exp_row = expected[r]
        cmp_row = computed[r]
        marks = []
        for c in range(cols):
            if exp_row[c] != cmp_row[c]:
                marks.append("^")
                mismatch_coords.append((r, c, exp_row[c], cmp_row[c]))
            else:
                marks.append(" ")
        out.append("E: " + "".join(str(x) for x in exp_row))
        out.append("C: " + "".join(str(x) for x in cmp_row))
        out.append("   " + "".join(marks))
    if len(expected) != len(computed) or len(expected[0]) != len(computed[0]):
        out.append(f"[Shape differs] expected {len(expected)}x{len(expected[0])} vs {len(computed)}x{len(computed[0])}")
    summary = f"Mismatches: {len(mismatch_coords)}"
    if mismatch_coords:
        sample = ", ".join(f"(r{r} c{c} {e}->{c2})" for r,c,e,c2 in mismatch_coords[:10])
        if len(mismatch_coords) > 10:
            sample += " ..."
        summary += "  " + sample
    out.append(summary)
    return "\n".join(out)

def evaluate_batch(
    model,
    tokenizer,
    dataset,
    dataset_formatted,
    start_idx: int = 0,
    num_examples: int = 10,
    verbose: bool = True,
    per_example_timeout: int = 30,
    show_grids: bool = True,
    show_generated_code: bool = True,
    show_gold_code: bool = True,
    show_diff: bool = True
):
    """
    Prints generated code, correct (gold) code, full grids, and diffs.
    """
    results = []
    passed = 0
    skipped = 0
    end_idx = min(start_idx + num_examples, len(dataset))

    for idx in tqdm(range(start_idx, end_idx), desc="Evaluating batch"):
        if idx % 10 == 0:
            torch.cuda.empty_cache()
        example = dataset[idx]

        try:
            with timeout_context(per_example_timeout):
                msgs = dataset_formatted[idx]['messages']
                if verbose:
                    print(f"\n{'='*100}")
                    print(f"Example {idx} ({idx-start_idx+1}/{num_examples})")

                prompt = tokenizer.apply_chat_template(
                    [msgs[0]],
                    tokenize=False,
                    add_generation_prompt=True
                )
                inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=128,
                        temperature=0.1,
                        do_sample=True,
                        eos_token_id=tokenizer.eos_token_id,
                        pad_token_id=tokenizer.pad_token_id
                    )

                model_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
                generated_code = extract_output_grid_line(model_output)

                del inputs, outputs
                torch.cuda.empty_cache()

                success, computed_output, error_msg = interpreter.test_solution(
                    generated_code,
                    example['test_input'],
                    example['test_output']
                )

                gold_code = example.get('solution')
                expected_output = example['test_output']

                if verbose:
                    if show_generated_code:
                        print("\n--- Generated Code ---")
                        print(generated_code)
                    if show_gold_code:
                        print("\n--- Correct (Gold) Code ---")
                        print(gold_code if gold_code is not None else "N/A")
                    if show_grids:
                        print("\n--- Expected Output Grid ---")
                        print(pretty_grid(expected_output))
                        print("\n--- Computed Output Grid ---")
                        print(pretty_grid(computed_output))
                    if not success and show_diff:
                        print("\n--- DIFF ---")
                        print(diff_grids(expected_output, computed_output))
                    print("\n✓ PASSED" if success else f"\n✗ FAILED: {error_msg}")

                if success:
                    passed += 1

                results.append({
                    'idx': idx,
                    'success': success,
                    'generated_code': generated_code,
                    'gold_code': gold_code,
                    'expected_output_grid': expected_output,
                    'computed_output_grid': computed_output,
                    'error': None if success else error_msg
                })

        except TimeoutException:
            if verbose:
                print(f"\n⚠️ TIMEOUT: Example {idx} > {per_example_timeout}s")
            skipped += 1
            results.append({
                'idx': idx,
                'success': False,
                'generated_code': "TIMEOUT",
                'gold_code': example.get('solution'),
                'expected_output_grid': example.get('test_output'),
                'computed_output_grid': None,
                'error': f"Timeout > {per_example_timeout}s"
            })
            torch.cuda.empty_cache()

        except Exception as e:
            if verbose:
                print(f"\n❌ ERROR: Example {idx} crashed: {e}")
            skipped += 1
            results.append({
                'idx': idx,
                'success': False,
                'generated_code': "ERROR",
                'gold_code': example.get('solution', 'Unknown'),
                'expected_output_grid': example.get('test_output'),
                'computed_output_grid': None,
                'error': str(e)
            })
            torch.cuda.empty_cache()

    processed = (end_idx - start_idx) - skipped
    accuracy = passed / processed if processed > 0 else 0.0
    print(f"\n{'='*100}")
    print(f"FINAL: {passed}/{processed} passed ({skipped} skipped)  Acc: {accuracy:.2%}")
    return results, accuracy

# Example:
# results, acc = evaluate_batch(model, tokenizer, test_dataset, test_dataset_formatted, num_examples=

In [ ]:
# Run batch evaluation
results, accuracy = evaluate_batch(
    model,
    tokenizer,
    test_dataset,
    test_dataset_formatted,
    start_idx=0,
    num_examples=10,
    verbose=False
)

In [ ]:
full_results, full_accuracy = evaluate_batch(
    model,
    tokenizer,
    test_dataset,
    test_dataset_formatted,
    start_idx=0,
    num_examples=len(test_dataset),
    verbose=False
)
print(f"\nFull dataset accuracy: {full_accuracy:.2%}")

Evaluating batch:   0%|          | 0/240 [00:00<?, ?it/s]